# Module 02 — Demand Modeling & Uncertainty-Aware Forecasting

This notebook certifies demand models for use in an automated pricing system.

The goal is not accuracy alone, but:

- economic validity

- causal consistency

- uncertainty awareness

- cold-start safety

- system-level reliability

Only models that pass statistical, economic, and operational gates are allowed to downstream bandit optimization (Module 03).

## Problem Definition 

We model booking probability:

$$ 𝑃(𝑌 = 1 ∣ 𝑝,𝑥,𝑡) $$

Where:

- 𝑝: price

- 𝑥: listing & neighborhood attributes

- 𝑡: time (seasonality)

#### Business Objective

Estimate demand such that:

- Price elasticity is **negative**

- Predictions are **calibrated**

- Uncertainty shrinks correctly with aggregation

- Cold-start behavior is safe

In [2]:
import sys
import os
import numpy as np
import pandas as pd
import time
import logging
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.append(os.path.abspath(".."))

import tensorflow as tf
tf.random.set_seed(42)

from sklearn.metrics import (
    mean_squared_error,
    log_loss,
    roc_auc_score
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LinearRegression

from pricing_engine.data_loader import load_and_clean_seattle_data
from pricing_engine.demand_model import (
    HierarchicalBayesianLogit,
    LGBMTweedie,
    TFLatticeModel,
    DeepFMModel
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(name)s | %(levelname)s | %(message)s"
)


logger = logging.getLogger("DemandModeling")



#### Reproducibility & Execution Guarantees

All experiments in this notebook are deterministic and reproducible.
Randomness is explicitly controlled, and all preprocessing steps are auditable.


### Data & Causality Lock
If the data violates economics, no model can fix it.


In [3]:
try:
    SCRIPT_DIR = Path(__file__).parent
except NameError:
    SCRIPT_DIR = Path.cwd()

PROJECT_ROOT = SCRIPT_DIR.parent

CALENDAR_PATH = PROJECT_ROOT / "data" / "calendar.csv"
LISTINGS_PATH = PROJECT_ROOT / "data" / "listings.csv"


In [4]:

PROJECT_ROOT = Path.cwd().parent
df_raw = load_and_clean_seattle_data(
    CALENDAR_PATH,
    LISTINGS_PATH,
)

logger.info(f"Rows loaded: {len(df_raw):,}")
assert not df_raw.empty, "Dataset is empty"

2026-01-04 20:02:49,385 | PriceEngine_Loader | INFO | Loading raw data...
2026-01-04 20:02:54,951 | PriceEngine_Loader | INFO | Total rows loaded: 1393570
2026-01-04 20:02:54,955 | PriceEngine_Loader | INFO | Action observed rate: 67.06%
2026-01-04 20:02:54,959 | PriceEngine_Loader | INFO | Exposure rate: 67.06%
2026-01-04 20:02:54,962 | PriceEngine_Loader | INFO | Booked proxy rate: 32.94%
2026-01-04 20:02:54,979 | DemandModeling | INFO | Rows loaded: 1,393,570


#### Sparsity Gate

Booking feedback must be sufficiently dense.
Sparse feedback destabilizes:

- Bayesian posteriors

- Bandit exploration

In [5]:
booking_rate = df_raw["action_observed"].mean()
logger.info(f"Booking rate: {booking_rate:.3f}")

assert booking_rate > 0.60, "❌ Sparsity gate failed"


2026-01-04 20:04:17,762 | DemandModeling | INFO | Booking rate: 0.671


#### Temporal Aggregation

We aggregate daily → weekly to:

- reduce noise

- stabilize elasticity

- align with pricing cadence


In [7]:
# --------------------------------------------------
# Gate 1.2: Aggregation & Causal Slice
# --------------------------------------------------
weekly_df = (
    df_raw
    # 1. Create Week Period
    .assign(week_period=pd.to_datetime(df_raw["date"]).dt.to_period("W").dt.start_time)
    
    .groupby(["listing_id", "week_period"])
    .agg(
        # Target Name = (Source Column, Function)
        is_booked=("is_booked_proxy", "max"),  # Use MAX to capture 'any booking'
        avg_price=("price", "mean"),           # Mean price over exposed days
        
        # Diagnostics
        exposure_days=("exposed", "sum"),
        
        # Static Attributes (First value is fine as they are static)
        accommodates=("accommodates", "first"),
        bedrooms=("bedrooms", "first"),
        bathrooms=("bathrooms", "first"),
        neighborhood=("neighborhood", "first")
    )
    .reset_index()
)

# 2. Filter for Validity (Causal Slice)
# We only keep weeks where the listing was actually exposed (available to be booked)
# Otherwise, price is undefined/imputed, which breaks causality.
weekly_df = weekly_df[weekly_df["exposure_days"] > 0].copy()

# 3. Rename for consistency
weekly_df.rename(columns={"week_period": "week_date"}, inplace=True)

# 4. Critical Gate Check
print(f"Weekly Rows: {len(weekly_df):,}")
assert weekly_df.shape[0] > 100_000, "❌ Gate 1.2 Failed: Aggregation resulted in too few rows."
print("✅ Gate 1.2 Passed: Causal aggregation successful.")

weekly_df.head()

Weekly Rows: 141,080
✅ Gate 1.2 Passed: Causal aggregation successful.


,listing_id,week_date,is_booked,avg_price,exposure_days,accommodates,bedrooms,bathrooms,neighborhood
8,3335,2016-02-29,0,120.0,7,4,2.0,1.0,Rainier Valley
9,3335,2016-03-07,0,120.0,7,4,2.0,1.0,Rainier Valley
10,3335,2016-03-14,0,120.0,7,4,2.0,1.0,Rainier Valley
11,3335,2016-03-21,0,120.0,7,4,2.0,1.0,Rainier Valley
12,3335,2016-03-28,0,120.0,7,4,2.0,1.0,Rainier Valley


#### Leakage Detection

Leakage creates illusory accuracy and false elasticity.

In [8]:
leak_features = ["accommodates", "bedrooms", "bathrooms"]
corrs = weekly_df[leak_features + ["is_booked"]].corr()["is_booked"]

corrs


accommodates    0.059504
bedrooms        0.031447
bathrooms       0.015580
is_booked       1.000000
Name: is_booked, dtype: float64

In [9]:
assert corrs.drop("is_booked").abs().max() < 0.9, "❌ Leakage detected"


### Baseline Economics

Before ML, we verify economics.

#### Price–Demand Slope

Expected relationship:

$$ ∂𝐸[𝑌]/∂𝑝 < 0 $$


In [10]:
# Linear Sanity Check

X = weekly_df[["avg_price"]]
y = weekly_df["is_booked"]

lin = LinearRegression().fit(X, y)
slope = lin.coef_[0]

slope


-6.8742181790503935e-06

In [11]:
assert slope < 0, "❌ Positive price slope"
assert abs(slope - (-0.238)) < 0.1, "❌ Audit mismatch"


AssertionError: ❌ Audit mismatch

Elasticity Prior (Markdown)

From Module 01 (Causal DML):

𝛽
𝑝
𝑟
𝑖
𝑐
𝑒
∼
𝑁
(
−
0.0323
,
𝜎
2
)


This is injected, not learned.

In [12]:

CAUSAL_BETA_PRIOR = -0.0323

### Model Certification Loop
| Model              | Role                       |
| ------------------ | -------------------------- |
| Hierarchical Bayes | Uncertainty + cold start   |
| LGBM Tweedie       | High accuracy              |
| TF Lattice         | Guaranteed monotonicity    |
| DeepFM             | Expressive (research only) |


In [13]:
MODELS = [
    HierarchicalBayesianLogit(beta_prior=CAUSAL_BETA_PRIOR),
    LGBMTweedie(),
    TFLatticeModel(),
    DeepFMModel()
]


In [ ]:

SPLIT_DATE = "2016-09-01"

train_df = weekly_df[weekly_df["week_date"] < SPLIT_DATE].copy()
test_df  = weekly_df[weekly_df["week_date"] >= SPLIT_DATE].copy()

logger.info(f"Training on {len(train_df):,} rows. Testing on {len(test_df):,} rows.")
assert len(train_df) > 0 and len(test_df) > 0, "❌ Split failed: One partition is empty."



global_mean = train_df["is_booked"].mean()


Train Rows: 92,065 (65.3%)
Test Rows:  49,015 (34.7%)
✅ Train/Test Split Successful.


#### Certification Loop

Each model must pass all gates to be promoted.

In [20]:
ML_FEATURES = [
    "log_price", 
    "week_of_year", 
    "month", 
    "accommodates", 
    "bedrooms", 
    "bathrooms", 
    "neighborhood" # Categorical
]

TARGET = "is_booked"

In [21]:

certified_models = {}

for model in MODELS:
    logger.info(f"\n🔍 Evaluating {model.name}...")
    
    # --- FIX: Pass features and target explicitly ---
    try:
        start_time = time.time()
        
        # Train
        model.fit(train_df, features=ML_FEATURES, target=TARGET)
        train_time = time.time() - start_time
        
        # Predict
        preds = model.predict(test_df)
        
        # Evaluate
        metrics = model.evaluate(test_df, target=TARGET)
        
        logger.info(f"   ✅ Trained in {train_time:.2f}s | Metrics: {metrics}")
        
        # Store for next gates
        certified_models[model.name] = {
            "model": model,
            "metrics": metrics,
            "status": "CANDIDATE" 
        }
        
    except Exception as e:
        logger.error(f"   ❌ Failed: {str(e)}")
        certified_models[model.name] = {"status": "FAILED", "error": str(e)}

2026-01-04 20:46:27,564 | DemandModeling | INFO | 
🔍 Evaluating HierarchicalBayes...
2026-01-04 20:46:27,596 | DemandModeling | ERROR |    ❌ Failed: 'log_price'
2026-01-04 20:46:27,596 | DemandModeling | INFO | 
🔍 Evaluating LGBM_Tweedie...
2026-01-04 20:46:27,615 | DemandModeling | ERROR |    ❌ Failed: "['log_price', 'week_of_year', 'month'] not in index"
2026-01-04 20:46:27,615 | DemandModeling | INFO | 
🔍 Evaluating TF_Lattice...
2026-01-04 20:46:27,721 | DemandModeling | ERROR |    ❌ Failed: 'log_price'
2026-01-04 20:46:27,721 | DemandModeling | INFO | 
🔍 Evaluating DeepFM...
2026-01-04 20:46:27,726 | DemandModeling | ERROR |    ❌ Failed: "['log_price', 'week_of_year', 'month'] not in index"


#### Statistical Fit Gate 

Metrics depend on output type:

- Probability → LogLoss, AUC

- Expectation → RMSE

In [ ]:
if model.prediction_type == "expectation":
    rmse = np.sqrt(mean_squared_error(test_df["is_booked"], preds))
    baseline_rmse = np.sqrt(mean_squared_error(
        test_df["is_booked"],
        np.full(len(test_df), global_mean)
    ))
    assert rmse < baseline_rmse
else:
    ll = log_loss(test_df["is_booked"], preds)
    auc = roc_auc_score(test_df["is_booked"], preds)
    baseline_ll = log_loss(
        test_df["is_booked"],
        np.full(len(test_df), global_mean)
    )
    assert ll < baseline_ll


#### Monotonicity Gate 

Economic law:

$$ 𝑝↑⇒𝐷↓ $$


In [ ]:
probe = pd.DataFrame({
    "avg_price": [50, 100, 150],
    "log_price": np.log1p([50,100,150])
})

probe_preds = model.predict(probe)

if not (probe_preds[0] >= probe_preds[1] >= probe_preds[2]):
    logger.warning("Monotonicity violated → RESEARCH_ONLY")
    continue


#### Calibration Gate

Well-calibrated probabilities prevent:

- overpricing

- revenue collapse

In [ ]:
prob_true, prob_pred = calibration_curve(
    test_df["is_booked"], preds, n_bins=10
)

ece = np.mean(np.abs(prob_true - prob_pred))
assert ece < 0.15


Cold Start Test

Unknown listings must:

- not crash

- regress toward mean

In [ ]:
cold_id = test_df["listing_id"].max()
cold_probe = test_df[test_df["listing_id"] == cold_id][:1]

cold_pred = model.predict(cold_probe)[0]
assert np.isfinite(cold_pred)


### Safety & Uncertainty

#### Bayesian Shrinkage

Correct uncertainty ordering:

$$ 𝜎(listing)  >𝜎(neighborhood) $$
	​


In [ ]:
if isinstance(model, HierarchicalBayesianLogit):
    alphas = model.alpha_map
    assert np.std(alphas["neighborhood"]) < np.std(alphas["listing"])


#### Safety Envelope

In [ ]:
full_preds = model.predict(weekly_df)
assert 0.0001 <= full_preds.mean() <= 0.999


##### Golden Record Regression Test 

Protects against silent model drift.

In [ ]:
GOLDEN_ID = 3335
REFERENCE_PROB = 0.075

probe = pd.DataFrame([{
    "listing_id": GOLDEN_ID,
    "avg_price": 120.0,
    "log_price": np.log1p(120.0),
    "accommodates": 4,
    "bedrooms": 1.0,
    "bathrooms": 1.0
}])

pred = model.predict(probe)[0]
assert abs(pred - REFERENCE_PROB) < REFERENCE_PROB * 0.2


## System Sign-off

In [ ]:
start = time.time()
_ = model.predict(weekly_df.sample(10_000))
throughput = 10_000 / (time.time() - start)

assert throughput > 1_000


In [ ]:
certified_models[model.name] = model

import joblib
joblib.dump(certified_models, "demand_models.pkl")
